Importando as bibliotecas

In [26]:
import pandas as pd
import numpy as np
import uuid6
import psycopg2
import os
import io
from dotenv import load_dotenv

Extraindo os dataframes

In [27]:
clientes = pd.read_csv("../data/clientes.csv")
pedidos = pd.read_csv("../data/pedidos.csv")

Vamos trabalhar primeiro com a tabela clientes

Análise panorâmica:

In [28]:
clientes.head()

,cliente_id,nome,email,telefone,cidade,estado,data_cadastro,segmento,limite_credito
0,1,João Silva,joao.silva@gmail.com,(11) 98765-4321,São Paulo,SP,2021-03-15,Varejo,5000.0
1,2,Maria Souza,MARIA.SOUZA@HOTMAIL.COM,11 99234-5678,são paulo,SP,2021-04-02,Atacado,15000.0
2,3,Pedro Alves,pedro.alves@yahoo.com,(21)98001-2345,Rio de Janeiro,RJ,2021-04-10,Varejo,3500.0
3,4,Ana Costa,NaN,21 97654-3210,rio de janeiro,RJ,2021-05-01,Varejo,3500.0
4,5,Carlos Mendes,carlos.mendes@empresa.com,(31) 96543-2109,Belo Horizonte,MG,2021-05-20,Atacado,20000.0


In [29]:
clientes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   cliente_id      40 non-null     int64  
 1   nome            40 non-null     object 
 2   email           39 non-null     object 
 3   telefone        37 non-null     object 
 4   cidade          40 non-null     object 
 5   estado          40 non-null     object 
 6   data_cadastro   40 non-null     object 
 7   segmento        40 non-null     object 
 8   limite_credito  40 non-null     float64
dtypes: float64(1), int64(1), object(7)
memory usage: 2.9+ KB


In [30]:
int(clientes.duplicated(subset=clientes.columns.drop("cliente_id")).sum())

0

Limpeza do dataset

In [31]:
clientes["telefone"] = clientes["telefone"].str.replace(r"[^0-9]", "", regex=True)

In [32]:
clientes = clientes.fillna("-")

In [33]:
# Colocando essas colunas em minúsculo pois é mais fácil de trabalhar
cols = ["nome","email","cidade","segmento"]

clientes[cols] = clientes[cols].apply(lambda x: x.str.lower()).astype(str)

In [34]:
clientes["data_cadastro"] = pd.to_datetime(clientes["data_cadastro"])

Agora vamos analisar a tabela pedidos

In [35]:
pedidos.sample(10)

,pedido_id,cliente_id,data_pedido,data_entrega,status,valor_total,desconto,forma_pagamento,canal,vendedor,cidade_entrega,observacao
62,1063,11,2023-08-11,2023-08-19,Entregue,25000.0,3750.0,Transferência,Televendas,Mariana,Brasília,NaN
22,1023,21,2023-03-18,2023-03-26,Entregue,960.0,0.0,PIX,Online,Ana Paula,Goiânia,NaN
9,1010,8,2023-02-03,2023-02-11,Entregue,920.0,0.0,PIX,Online,Mariana,Curitiba,NaN
0,1001,1,2023-01-05,2023-01-12,Entregue,1250.0,0.0,Cartão de Crédito,Online,Ana Paula,São Paulo,NaN
76,1077,7,2023-09-22,2023-09-30,Entregue,8700.0,870.0,Transferência,Online,Mariana,Curitiba,NaN
36,1037,35,2023-05-13,2023-05-21,Entregue,720.0,0.0,Cartão de Débito,Loja Física,Bruno,Teresina,NaN
49,1050,3,2023-07-03,2023-07-11,Entregue,850.0,0.0,PIX,Online,Ana Paula,Rio de Janeiro,NaN
26,1027,25,2023-04-06,2023-04-14,Entregue,22000.0,4400.0,Transferência,Televendas,Mariana,Campo Grande,NaN
29,1030,28,2023-04-15,2023-04-23,Entregue,3600.0,180.0,Boleto,Televendas,Carlos,Natal,NaN
86,1087,38,2023-10-22,2023-10-30,Entregue,1900.0,0.0,Cartão de Débito,Loja Física,Bruno,São Luís,NaN


In [36]:
pedidos.describe()

,pedido_id,cliente_id,valor_total,desconto
count,100.000000,100.000000,100.000000,100.000000
mean,1050.500000,19.920000,6085.400000,783.950000
std,29.011492,14.187546,8297.597002,1573.825888
min,1001.000000,0.000000,310.000000,0.000000
25%,1025.750000,8.750000,1072.500000,0.000000
50%,1050.500000,19.500000,2650.000000,127.500000
75%,1075.250000,29.000000,5900.000000,590.000000
max,1100.000000,99.000000,40000.000000,8000.000000


In [37]:
pedidos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 12 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   pedido_id        100 non-null    int64  
 1   cliente_id       100 non-null    int64  
 2   data_pedido      100 non-null    object 
 3   data_entrega     86 non-null     object 
 4   status           100 non-null    object 
 5   valor_total      100 non-null    float64
 6   desconto         100 non-null    float64
 7   forma_pagamento  100 non-null    object 
 8   canal            100 non-null    object 
 9   vendedor         100 non-null    object 
 10  cidade_entrega   100 non-null    object 
 11  observacao       9 non-null      object 
dtypes: float64(2), int64(2), object(8)
memory usage: 9.5+ KB


In [38]:
int(pedidos.duplicated(subset=pedidos.columns.drop("pedido_id")).sum())

0

Limpeza do dataset

In [39]:
pedidos = pedidos[(pedidos["cliente_id"]<=40) & (pedidos["cliente_id"]>=1) & (pedidos["status"] != "-1")]

In [40]:
pedidos = pedidos.fillna("-")

In [41]:
pedidos["data_pedido"] = pd.to_datetime(pedidos["data_pedido"], errors="ignore")
pedidos["data_entrega"] = pd.to_datetime(pedidos["data_entrega"], errors="ignore")

C:\Users\kauan\AppData\Local\Temp\ipykernel_23700\3638439585.py:1: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  pedidos["data_pedido"] = pd.to_datetime(pedidos["data_pedido"], errors="ignore")
C:\Users\kauan\AppData\Local\Temp\ipykernel_23700\3638439585.py:2: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_datetime without passing `errors` and catch exceptions explicitly instead
  pedidos["data_entrega"] = pd.to_datetime(pedidos["data_entrega"], errors="ignore")


In [42]:
# Colocando essas colunas em minúsculo pois é mais fácil de trabalhar
cols = ["status","forma_pagamento","canal","vendedor","cidade_entrega","observacao"]

pedidos[cols] = pedidos[cols].apply(lambda x: x.str.lower()).astype(str)

Carregamento do dataset na camada silver do PostgreSQL

In [43]:
buffer1 = io.StringIO()
clientes.to_csv(buffer1, index=False)
buffer1.seek(0)

buffer2 = io.StringIO()
pedidos.to_csv(buffer2, index=False)
buffer2.seek(0)

0

In [44]:
load_dotenv(dotenv_path="../../.env")

conn = psycopg2.connect(
    host=os.getenv("host"),
    dbname=os.getenv("dbname"),
    user=os.getenv("user"),
    password=os.getenv("password"),
    port=os.getenv("port")
)

In [45]:
pedidos.head(0)

,pedido_id,cliente_id,data_pedido,data_entrega,status,valor_total,desconto,forma_pagamento,canal,vendedor,cidade_entrega,observacao


In [46]:
cur = conn.cursor()

cur.execute("CREATE SCHEMA IF NOT EXISTS case_3;")

cur.execute("""
CREATE TABLE IF NOT EXISTS case_3.clientes_silver (
    cliente_id INTEGER PRIMARY KEY,
    nome VARCHAR(255),
    email VARCHAR(255),
    telefone VARCHAR(20),
    cidade VARCHAR(60),
    estado VARCHAR(2),
    data_cadastro TIMESTAMP,
    segmento VARCHAR(40),
    limite_credito DECIMAL(20, 2)
);
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS case_3.pedidos_silver (
    pedido_id INTEGER PRIMARY KEY,
    cliente_id INTEGER REFERENCES case_3.clientes_silver(cliente_id),
    data_pedido TIMESTAMP,
    data_entrega VARCHAR(11),
    status VARCHAR(60),
    valor_total DECIMAL(20, 2),
    desconto DECIMAL(20, 2),
    forma_pagamento VARCHAR(60),
    canal VARCHAR(60),
    vendedor VARCHAR(255),
    cidade_entrega VARCHAR(60),
    observacao VARCHAR(400)
);
""")

cur.execute("TRUNCATE TABLE case_3.pedidos_silver;")
cur.execute("TRUNCATE TABLE case_3.clientes_silver CASCADE;")

buffer1.seek(0)
buffer2.seek(0)

cur.copy_expert(sql="COPY case_3.clientes_silver FROM STDIN WITH CSV HEADER", file=buffer1)
cur.copy_expert(sql="COPY case_3.pedidos_silver FROM STDIN WITH CSV HEADER", file=buffer2)

conn.commit() 
cur.close() 
conn.close()
